# Principio Abierto/Cerrado (OCP)

El software debe estar abierto a extensión, pero cerrado a modificación.

## Sin aplicar OCP

Cada tipo nuevo obliga a modificar el `if/elif` de la clase existente.

In [1]:
class CalculadoraDescuento:
    def __init__(self, tipo_cliente, porcentaje_base=0.0):
        self.tipo_cliente = tipo_cliente; self.porcentaje_base = porcentaje_base
    def porcentaje(self):
        if self.tipo_cliente == 'nuevo': return 0.05
        if self.tipo_cliente == 'frecuente': return 0.15
        if self.tipo_cliente == 'corporativo': return 0.20
        return self.porcentaje_base
    def calcular(self, subtotal):
        return subtotal * (1 - self.porcentaje())

class PedidoConDescuento:
    def __init__(self, cliente, subtotal):
        self.cliente = cliente; self.subtotal = subtotal
    def describir(self):
        return f'Pedido de {self.cliente}: ${self.subtotal:,.0f}'
    def cobrar(self, calculadora):
        return calculadora.calcular(self.subtotal)

pedido_mal = PedidoConDescuento('Carlos', 100000)
print(pedido_mal.describir()); print(pedido_mal.cobrar(CalculadoraDescuento('frecuente')))
print("Agregar 'cumpleaños' requiere editar porcentaje() y sumar otro condicional.")

Pedido de Carlos: $100,000
85000.0
Agregar 'cumpleaños' requiere editar porcentaje() y sumar otro condicional.


## Aplicando OCP

El calculador acepta cualquier política. `DescuentoCumpleanos` demuestra una extensión nueva sin modificar `CalculadorTotal`.

In [ ]:
from abc import ABC, abstractmethod

class PoliticaDescuento(ABC):
    def __init__(self, nombre, descripcion):
        self.nombre = nombre; self.descripcion = descripcion
    @abstractmethod
    def porcentaje(self, subtotal): pass
    def explicar(self): return f'{self.nombre}: {self.descripcion}'

class DescuentoFrecuente(PoliticaDescuento):
    def __init__(self, compras_minimas=5, tasa=0.15):
        super().__init__('Frecuente', 'Premia recurrencia'); self.compras_minimas = compras_minimas; self.tasa = tasa
    def porcentaje(self, subtotal): return self.tasa if subtotal > 0 else 0.0
    def explicar(self): return f'{super().explicar()} ({self.tasa:.0%})'

class DescuentoCumpleanos(PoliticaDescuento):
    def __init__(self, mes_actual, mes_nacimiento):
        super().__init__('Cumpleaños', 'Beneficio mensual'); self.mes_actual = mes_actual; self.mes_nacimiento = mes_nacimiento
    def porcentaje(self, subtotal): return 0.25 if self.mes_actual == self.mes_nacimiento and subtotal > 0 else 0.0
    def explicar(self): return f'{super().explicar()} (mes {self.mes_nacimiento})'

class CalculadorTotal:
    def __init__(self, moneda='COP', redondear=True):
        self.moneda = moneda; self.redondear = redondear
    def calcular(self, subtotal, politica):
        total = subtotal * (1 - politica.porcentaje(subtotal)); return round(total) if self.redondear else total
    def detalle(self, subtotal, politica):
        return {'política': politica.nombre, 'total': self.calcular(subtotal, politica), 'moneda': self.moneda}

calculador = CalculadorTotal()
for politica in (DescuentoFrecuente(), DescuentoCumpleanos(8, 8)): print(calculador.detalle(100000, politica))
print('La política nueva funcionó sin editar CalculadorTotal.')

El punto de extensión es `PoliticaDescuento`: las reglas crecen mediante nuevas implementaciones, mientras el calculador permanece estable.